## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("AnaliseDadosInstallments") \
    .master("spark://spark-master:7077") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/13 14:48:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
installments_payments = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/installments_payments.csv")
installments_payments.createOrReplaceTempView("installments_payments")

installments_payments.show(5, truncate=False)

+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|1054186   |161674    |1.0                   |6                    |-1180.0        |-1187.0           |6948.36       |6948.36    |
|1330831   |151639    |0.0                   |34                   |-2156.0        |-2156.0           |1716.525      |1716.525   |
|2085231   |193053    |2.0                   |1                    |-63.0          |-63.0             |25425.0       |25425.0    |
|2452527   |199697    |1.0                   |3                    |-2418.0        |-2426.0           |24350.13      |24350.13   |
|2714724   |167756    |1.0                   |2                    |-1383.0        

In [3]:
installments_payments.count()

13605401

In [16]:
installments_payments = spark.sql("""
    SELECT
        SK_ID_PREV as PK_AGG_INSTALLMENTS,
        SK_ID_CURR,
        DAYS_ENTRY_PAYMENT as FVL_DAYS_ENTRY_PAYMENT_INSTALM,
        AMT_INSTALMENT as FVL_AMT_INSTALMENT_INSTALM,
        AMT_PAYMENT as FVL_AMT_PAYMENT_INSTALM,
        cast(date_add('2023-12-01', cast(DAYS_INSTALMENT as int)) as timestamp) as PK_DATREF_INSTALM,
        substr(translate(cast(date_add('2023-12-01', cast(DAYS_INSTALMENT as int)) as string),'-',''),1,6) as PK_DATPART_INSTALM,
        DAYS_INSTALMENT
    FROM 
        installments_payments
""")
# Retirando valores nulos
installments_payments = installments_payments.where(col("DAYS_INSTALMENT").isNotNull())

# Filtrando somente histórico necessário (15 meses)
stage01 = installments_payments.where(col("DAYS_INSTALMENT") >= -2922)
stage01 = stage01.drop("DAYS_INSTALMENT")

stage01.createOrReplaceTempView("stage01")
stage01.count()

13605401

In [17]:
stage01.show(5, truncate=False)

+-------------------+----------+------------------------------+--------------------------+-----------------------+-------------------+------------------+
|PK_AGG_INSTALLMENTS|SK_ID_CURR|FVL_DAYS_ENTRY_PAYMENT_INSTALM|FVL_AMT_INSTALMENT_INSTALM|FVL_AMT_PAYMENT_INSTALM|PK_DATREF_INSTALM  |PK_DATPART_INSTALM|
+-------------------+----------+------------------------------+--------------------------+-----------------------+-------------------+------------------+
|1054186            |161674    |-1187.0                       |6948.36                   |6948.36                |2020-09-07 00:00:00|202009            |
|1330831            |151639    |-2156.0                       |1716.525                  |1716.525               |2018-01-05 00:00:00|201801            |
|2085231            |193053    |-63.0                         |25425.0                   |25425.0                |2023-09-29 00:00:00|202309            |
|2452527            |199697    |-2426.0                       |24350.13     

In [18]:
spark.sql("""
            select
                PK_DATPART_INSTALM,
                count(*) as Volume
            from stage01
            group by 1
            order by  1 desc
""").show(200)

+------------------+------+
|PK_DATPART_INSTALM|Volume|
+------------------+------+
|            202311|221822|
|            202310|280460|
|            202309|282377|
|            202308|300538|
|            202307|307269|
|            202306|299880|
|            202305|308954|
|            202304|296361|
|            202303|302290|
|            202302|268452|
|            202301|292147|
|            202212|285710|
|            202211|270552|
|            202210|274364|
|            202209|259517|
|            202208|261302|
|            202207|253096|
|            202206|237333|
|            202205|237810|
|            202204|222862|
|            202203|222971|
|            202202|194207|
|            202201|206874|
|            202112|197951|
|            202111|183592|
|            202110|182910|
|            202109|170594|
|            202108|170643|
|            202107|164862|
|            202106|154633|
|            202105|155271|
|            202104|145671|
|            202103|

In [19]:
spark.sql("""
            select
                PK_DATPART_INSTALM,
                count(*) as Volume
            from stage01
            group by 1
            order by  1 desc
""").count()

96

#### Salvando tabela particionada (Parquet)

In [20]:
nm_path = '/data/processed/installments/'
stage01.write.partitionBy('PK_DATPART_INSTALM').parquet(nm_path, mode='overwrite')

In [21]:
spark.stop()